<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/MEDICINE_TOPO_GEMMA_REASONING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://openrouter.ai/settings/credits

https://huggingface.co/frankmorales2020/gemma-4-e4b-stl10-topo-2026

In [1]:
!pip install -q transformers torch openai python-dotenv pyyaml requests numpy pydantic fastapi uvicorn
!pip install -U bitsandbytes>=0.46.1 -q
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 146.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [1]:
!pip install -q anthropic

In [2]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

print("Models available to your API key:\n")
for model in client.models.list(limit=100).data:
    print(f"  {model.id:<35} {model.display_name}")

Models available to your API key:

  claude-opus-5                       Claude Opus 5
  claude-sonnet-5                     Claude Sonnet 5
  claude-fable-5                      Claude Fable 5
  claude-opus-4-8                     Claude Opus 4.8
  claude-opus-4-7                     Claude Opus 4.7
  claude-sonnet-4-6                   Claude Sonnet 4.6
  claude-opus-4-6                     Claude Opus 4.6
  claude-opus-4-5-20251101            Claude Opus 4.5
  claude-haiku-4-5-20251001           Claude Haiku 4.5
  claude-sonnet-4-5-20250929          Claude Sonnet 4.5
  claude-opus-4-1-20250805            Claude Opus 4.1


In [ ]:
import requests
import json

def list_openrouter_models():
    """
    Connects to the OpenRouter API to fetch the list of available models
    and prints key details for each model.
    """

    # OpenRouter API endpoint for listing models
    API_URL = "https://openrouter.ai/api/v1/models"

    print(f"Fetching models from: {API_URL}\n")

    try:
        # Send a GET request to the models endpoint
        response = requests.get(API_URL)

        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()

        # The list of models is contained in the 'data' field
        models = data.get('data', [])

        if not models:
            print("No models found in the API response.")
            return

        # --- Print the results in a formatted way ---
        print(f"Found {len(models)} total models. Key details:\n")
        print("{:<45} {:<30} {:<15}".format("Model ID", "Name", "Context Length"))
        print("-" * 90)

        # Iterate over the model list and print details
        for model in models:
            model_id = model.get('id', 'N/A')
            name = model.get('name', 'N/A')
            context_length = model.get('context_length', 'N/A')

            # Truncate model ID and name for clean printing
            display_id = model_id[:42] + '...' if len(model_id) > 45 else model_id
            display_name = name[:27] + '...' if len(name) > 30 else name

            print("{:<45} {:<30} {:<15}".format(display_id, display_name, context_length))

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while connecting to the OpenRouter API: {e}")
    except json.JSONDecodeError:
        print("Error: Failed to decode JSON response from the API.")

if __name__ == "__main__":
    list_openrouter_models()

## 🏥 Ferrari AI - Medical Diagnostics Agent - INKLING

In [1]:
# ============================================================
# FERRARI AI - MEDICAL DIAGNOSTICS AGENT
# Complete Agentic Solution for Clinical Decision Support
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
import random
import re

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if OPENROUTER_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{OPENROUTER_API_KEY[-4:]}")
    else:
        print("⚠️ OPENROUTER_API_KEY not found in Colab secrets.")
        OPENROUTER_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    OPENROUTER_API_KEY = ""

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ============================================================
# CORRECT INKLING MODEL ID
# ============================================================
INKLING_MODEL_ID = "thinkingmachines/inkling"
INKLING_MAX_TOKENS = 2048
INKLING_TEMPERATURE = 0.7

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ============================================================
# PART 2: INKLING CLIENT
# ============================================================

@dataclass
class InklingResponse:
    content: str
    reasoning: Optional[str] = None
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class InklingClient:
    def __init__(self,
                 api_key: Optional[str] = None,
                 model: str = INKLING_MODEL_ID,
                 max_tokens: int = INKLING_MAX_TOKENS,
                 temperature: float = INKLING_TEMPERATURE):
        self.api_key = api_key or OPENROUTER_API_KEY
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.base_url = "https://openrouter.ai/api/v1"

        if not self.api_key:
            print("⚠️ No API key found. Inkling will not work.")
        else:
            print(f"✅ Inkling client initialized")
            print(f"   Model: {model}")
            print(f"   Max Tokens: {max_tokens}")
            print(f"   Temperature: {temperature}")

        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def query(self,
              prompt: str,
              temperature: Optional[float] = None,
              max_tokens: Optional[int] = None) -> InklingResponse:
        if not self.api_key:
            return InklingResponse(
                content="ERROR: No API key. Please add OPENROUTER_API_KEY to Colab secrets."
            )

        temp = temperature if temperature is not None else self.temperature
        max_tok = max_tokens if max_tokens is not None else self.max_tokens

        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temp,
            "max_tokens": max_tok,
            "reasoning": {"enabled": True}
        }

        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60
            )

            if response.status_code != 200:
                error_content = f"API Error ({response.status_code}): {response.text[:200]}"
                return InklingResponse(content=error_content)

            data = response.json()
            message = data['choices'][0]['message']

            content = message.get('content')
            reasoning = message.get('reasoning')

            if not content and reasoning:
                content = reasoning
                reasoning = None

            if not content:
                content = "No response content received."

            return InklingResponse(
                content=content,
                reasoning=reasoning,
                raw_response=data,
                model=data.get('model')
            )

        except Exception as e:
            return InklingResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: MEDICAL DIAGNOSTICS AGENT
# ============================================================

class MedicalTaskType(Enum):
    SYMPTOM_CHECK = "symptom_check"
    DIAGNOSIS = "diagnosis"
    MEDICATION_CHECK = "medication_check"
    LAB_INTERPRETATION = "lab_interpretation"
    RISK_ASSESSMENT = "risk_assessment"
    TREATMENT_RECOMMENDATION = "treatment_recommendation"
    TRIAGE = "triage"
    GENERAL = "general"


@dataclass
class Patient:
    patient_id: str
    name: str
    age: int
    gender: str
    symptoms: List[str]
    existing_conditions: List[str]
    medications: List[str]
    allergies: List[str]
    vital_signs: Dict[str, float]
    lab_results: Dict[str, Any]
    visit_date: datetime

    def to_dict(self) -> Dict:
        return {
            'patient_id': self.patient_id,
            'name': self.name,
            'age': self.age,
            'gender': self.gender,
            'symptoms': self.symptoms,
            'existing_conditions': self.existing_conditions,
            'medications': self.medications,
            'allergies': self.allergies,
            'vital_signs': self.vital_signs,
            'lab_results': self.lab_results,
            'visit_date': self.visit_date.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class MedicalKnowledge:
    condition: str
    description: str
    common_symptoms: List[str]
    risk_factors: List[str]
    treatment_options: List[str]
    urgency_level: str  # LOW, MEDIUM, HIGH, CRITICAL


@dataclass
class MedicalResponse:
    task_type: MedicalTaskType
    action_taken: str
    data: Dict
    reasoning: str
    diagnosis: Optional[str] = None
    confidence: float = 0.0
    recommendations: List[str] = field(default_factory=list)
    referral_needed: bool = False
    urgency: str = "LOW"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class MedicalDiagnosticsAgent:
    """
    Agentic AI System for Medical Diagnostics and Clinical Decision Support
    Combines Gemma-4 classification with Inkling reasoning
    """

    def __init__(self):
        print("\n" + "="*60)
        print("🏥  MEDICAL DIAGNOSTICS AGENT")
        print("="*60)

        # Initialize Ferrari AI components
        self.gemma = GemmaTOPOCertified()
        self.inkling = InklingClient()

        # Medical database (simulated)
        self.patients: Dict[str, Patient] = {}
        self.medical_knowledge: Dict[str, MedicalKnowledge] = {}
        self._initialize_patient_data()
        self._initialize_medical_knowledge()

        print("="*60)
        print("✅ Medical Diagnostics Agent Ready!")
        print(f"   🏥 Patients: {len(self.patients)}")
        print(f"   📚 Conditions: {len(self.medical_knowledge)}")
        print("="*60 + "\n")

    def _initialize_patient_data(self):
        """Initialize simulated patient data."""
        now = datetime.now()

        patients_data = [
            (
                "P001", "John Smith", 45, "Male",
                ["fever", "cough", "shortness of breath"],
                ["hypertension", "type 2 diabetes"],
                ["lisinopril", "metformin"],
                ["penicillin"],
                {"bp": "140/90", "hr": 95, "temp": 38.5, "spo2": 94},
                {"wbc": "12.5", "crp": "15.2", "glucose": "140"},
                now - timedelta(hours=2)
            ),
            (
                "P002", "Sarah Johnson", 32, "Female",
                ["headache", "nausea", "photophobia"],
                ["migraine history"],
                ["sumatriptan"],
                ["sulfa drugs"],
                {"bp": "120/80", "hr": 72, "temp": 37.0, "spo2": 99},
                {"wbc": "7.2", "crp": "1.5", "glucose": "95"},
                now - timedelta(days=1)
            ),
            (
                "P003", "Robert Chen", 65, "Male",
                ["chest pain", "dizziness", "fatigue"],
                ["coronary artery disease", "hyperlipidemia"],
                ["atorvastatin", "aspirin", "metoprolol"],
                [],
                {"bp": "150/95", "hr": 88, "temp": 37.2, "spo2": 97},
                {"wbc": "9.8", "crp": "8.7", "glucose": "120", "troponin": "0.08"},
                now - timedelta(hours=4)
            ),
            (
                "P004", "Maria Garcia", 28, "Female",
                ["abdominal pain", "nausea", "vomiting"],
                ["GERD"],
                ["omeprazole"],
                ["latex"],
                {"bp": "115/75", "hr": 80, "temp": 37.5, "spo2": 98},
                {"wbc": "11.2", "crp": "5.8", "glucose": "90"},
                now - timedelta(hours=6)
            ),
            (
                "P005", "James Wilson", 55, "Male",
                ["fever", "chills", "muscle aches", "fatigue"],
                ["none"],
                [],
                [],
                {"bp": "135/85", "hr": 100, "temp": 39.0, "spo2": 96},
                {"wbc": "14.5", "crp": "22.3", "glucose": "110"},
                now - timedelta(hours=1)
            ),
        ]

        for patient_data in patients_data:
            patient = Patient(
                patient_id=patient_data[0],
                name=patient_data[1],
                age=patient_data[2],
                gender=patient_data[3],
                symptoms=patient_data[4],
                existing_conditions=patient_data[5],
                medications=patient_data[6],
                allergies=patient_data[7],
                vital_signs=patient_data[8],
                lab_results=patient_data[9],
                visit_date=patient_data[10]
            )
            self.patients[patient.patient_id] = patient

    def _initialize_medical_knowledge(self):
        """Initialize medical knowledge base."""
        knowledge_data = [
            MedicalKnowledge(
                condition="Pneumonia",
                description="Inflammation of the lungs typically caused by infection",
                common_symptoms=["fever", "cough", "shortness of breath", "chest pain"],
                risk_factors=["age > 65", "smoking", "immunosuppression", "chronic lung disease"],
                treatment_options=["Antibiotics", "Oxygen therapy", "Cough medicine"],
                urgency_level="HIGH"
            ),
            MedicalKnowledge(
                condition="Migraine",
                description="Recurrent headaches often with nausea and sensitivity to light",
                common_symptoms=["headache", "nausea", "photophobia", "aura"],
                risk_factors=["family history", "female gender", "stress"],
                treatment_options=["Pain relievers", "Triptans", "Preventive medications"],
                urgency_level="LOW"
            ),
            MedicalKnowledge(
                condition="Myocardial Infarction",
                description="Heart attack due to reduced blood flow to the heart",
                common_symptoms=["chest pain", "shortness of breath", "dizziness", "nausea"],
                risk_factors=["age", "hypertension", "high cholesterol", "smoking"],
                treatment_options=["Aspirin", "Nitroglycerin", "PCI", "Thrombolytics"],
                urgency_level="CRITICAL"
            ),
            MedicalKnowledge(
                condition="Gastroenteritis",
                description="Inflammation of the stomach and intestines",
                common_symptoms=["abdominal pain", "nausea", "vomiting", "diarrhea"],
                risk_factors=["Contaminated food/water", "Poor hygiene"],
                treatment_options=["Fluid replacement", "Anti-emetics", "Antidiarrheals"],
                urgency_level="LOW"
            ),
            MedicalKnowledge(
                condition="Sepsis",
                description="Life-threatening organ dysfunction caused by infection",
                common_symptoms=["fever", "chills", "confusion", "rapid heart rate"],
                risk_factors=["Elderly", "Immunocompromised", "Chronic illness"],
                treatment_options=["Antibiotics", "IV fluids", "Vasopressors"],
                urgency_level="CRITICAL"
            ),
        ]

        for knowledge in knowledge_data:
            self.medical_knowledge[knowledge.condition] = knowledge

    def identify_task(self, query: str) -> Dict:
        """Classify the medical query type."""
        query_lower = query.lower()

        task_mapping = {
            'symptom': MedicalTaskType.SYMPTOM_CHECK,
            'symptoms': MedicalTaskType.SYMPTOM_CHECK,
            'diagnose': MedicalTaskType.DIAGNOSIS,
            'diagnosis': MedicalTaskType.DIAGNOSIS,
            'medication': MedicalTaskType.MEDICATION_CHECK,
            'drug': MedicalTaskType.MEDICATION_CHECK,
            'lab': MedicalTaskType.LAB_INTERPRETATION,
            'test': MedicalTaskType.LAB_INTERPRETATION,
            'result': MedicalTaskType.LAB_INTERPRETATION,
            'risk': MedicalTaskType.RISK_ASSESSMENT,
            'treatment': MedicalTaskType.TREATMENT_RECOMMENDATION,
            'urgent': MedicalTaskType.TRIAGE,
            'emergency': MedicalTaskType.TRIAGE,
            'triage': MedicalTaskType.TRIAGE,
        }

        for keyword, task_type in task_mapping.items():
            if keyword in query_lower:
                return {'task_type': task_type, 'confidence': 0.9}

        return {'task_type': MedicalTaskType.GENERAL, 'confidence': 0.6}

    def _get_patient(self, patient_id: str) -> Optional[Dict]:
        """Get patient data by ID."""
        if patient_id in self.patients:
            return self.patients[patient_id].to_dict()
        return None

    def _find_patients_by_symptom(self, symptom: str) -> List[Dict]:
        """Find patients with specific symptoms."""
        results = []
        for patient in self.patients.values():
            if any(symptom.lower() in s.lower() for s in patient.symptoms):
                results.append(patient.to_dict())
        return results

    def _find_patients_by_condition(self, condition: str) -> List[Dict]:
        """Find patients with specific existing conditions."""
        results = []
        for patient in self.patients.values():
            if any(condition.lower() in c.lower() for c in patient.existing_conditions):
                results.append(patient.to_dict())
        return results

    def _get_condition_knowledge(self, condition: str) -> Optional[Dict]:
        """Get medical knowledge for a condition."""
        if condition in self.medical_knowledge:
            return self.medical_knowledge[condition].__dict__
        return None

    def process_query(self, query: str) -> MedicalResponse:
        """
        Main processing method for the Medical Diagnostics Agent.
        """
        print(f"\n🏥 Medical Query: {query}")
        print("-" * 50)

        # Step 1: Identify task type
        task_decision = self.identify_task(query)
        task_type = task_decision['task_type']
        confidence = task_decision['confidence']

        print(f"📋 Task Identification: {task_type.value} (confidence: {confidence:.2%})")

        # Step 2: Extract patient ID if mentioned
        # ✅ FIXED: Correct regex pattern for patient ID extraction
        patient_match = re.search(r'P\d{3}', query.upper())
        patient_id = patient_match.group(0) if patient_match else None

        # Extract condition/symptom if mentioned
        conditions = list(self.medical_knowledge.keys())
        mentioned_condition = None
        for condition in conditions:
            if condition.lower() in query.lower():
                mentioned_condition = condition
                break

        # Step 3: Retrieve data based on task type
        data = {}
        action_taken = ""
        diagnosis = None
        recommendations = []
        referral_needed = False
        urgency = "LOW"

        if task_type == MedicalTaskType.DIAGNOSIS or task_type == MedicalTaskType.SYMPTOM_CHECK:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    action_taken = f"Retrieved patient {patient_id}"

                    # Check for matching conditions
                    matching_conditions = []
                    for condition, knowledge in self.medical_knowledge.items():
                        if any(symptom.lower() in condition.lower() for symptom in patient_data['symptoms']):
                            matching_conditions.append(condition)
                        elif any(symptom.lower() in ' '.join(knowledge.common_symptoms).lower()
                                for symptom in patient_data['symptoms']):
                            matching_conditions.append(condition)

                    if matching_conditions:
                        data['possible_conditions'] = matching_conditions
                        data['condition_details'] = [
                            self._get_condition_knowledge(c) for c in matching_conditions[:3]
                        ]
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['all_patients'] = [p.to_dict() for p in self.patients.values()]
                action_taken = "Retrieved all patients"

        elif task_type == MedicalTaskType.LAB_INTERPRETATION:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    data['lab_results'] = patient_data.get('lab_results', {})
                    action_taken = f"Retrieved lab results for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['lab_results'] = []
                for patient in self.patients.values():
                    data['lab_results'].append({
                        'patient_id': patient.patient_id,
                        'name': patient.name,
                        'results': patient.lab_results
                    })
                action_taken = "Retrieved lab results for all patients"

        elif task_type == MedicalTaskType.MEDICATION_CHECK:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    data['medications'] = patient_data.get('medications', [])
                    data['allergies'] = patient_data.get('allergies', [])
                    action_taken = f"Retrieved medication information for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['medications_summary'] = {}
                for patient in self.patients.values():
                    data['medications_summary'][patient.patient_id] = {
                        'name': patient.name,
                        'medications': patient.medications,
                        'allergies': patient.allergies
                    }
                action_taken = "Retrieved medication information for all patients"

        elif task_type == MedicalTaskType.RISK_ASSESSMENT:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    # Calculate risk factors
                    risk_factors = []
                    if patient_data['age'] > 65:
                        risk_factors.append("Age > 65")
                    if patient_data.get('existing_conditions'):
                        risk_factors.append("Existing conditions")
                    if patient_data.get('vital_signs', {}).get('bp'):
                        bp = patient_data['vital_signs']['bp']
                        if bp != "Normal":
                            risk_factors.append(f"Abnormal blood pressure: {bp}")
                    data['risk_factors'] = risk_factors
                    action_taken = f"Assessed risk factors for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['risk_assessment'] = "Please specify a patient ID"
                action_taken = "Risk assessment requires patient ID"

        elif task_type == MedicalTaskType.TRIAGE:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    # Determine urgency
                    vital_signs = patient_data.get('vital_signs', {})
                    symptoms = patient_data.get('symptoms', [])

                    if any(s in ['chest pain', 'shortness of breath'] for s in symptoms):
                        urgency = "CRITICAL"
                        referral_needed = True
                    elif 'fever' in symptoms and any(t in symptoms for t in ['chills', 'muscle aches']):
                        urgency = "HIGH"
                    elif 'headache' in symptoms:
                        urgency = "MEDIUM"
                    else:
                        urgency = "LOW"

                    data['triage_level'] = urgency
                    action_taken = f"Triaged patient {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['triage_summary'] = [
                    {'patient_id': p.patient_id, 'name': p.name, 'symptoms': p.symptoms}
                    for p in self.patients.values()
                ]
                action_taken = "Retrieved triage summary for all patients"

        else:  # GENERAL
            data['message'] = "General medical inquiry received"
            action_taken = "General query processed"

        # Step 4: Generate reasoning using Inkling
        reasoning_prompt = self._generate_reasoning_prompt(query, task_type, data)
        inkling_response = self.inkling.query(reasoning_prompt)

        print(f"🧠 Inkling Reasoning: {inkling_response.content[:200]}...")

        # Step 5: Build response
        return MedicalResponse(
            task_type=task_type,
            action_taken=action_taken,
            data=data,
            reasoning=inkling_response.content,
            diagnosis=mentioned_condition or "Further analysis needed",
            confidence=confidence,
            recommendations=self._extract_recommendations(inkling_response.content),
            referral_needed=referral_needed,
            urgency=urgency,
            timestamp=datetime.now().isoformat()
        )

    def _generate_reasoning_prompt(self, query: str, task_type: MedicalTaskType, data: Dict) -> str:
        """Generate a prompt for Inkling to reason about the medical situation."""
        return f"""
        You are a Medical Diagnostics AI Assistant providing clinical decision support.

        User Query: "{query}"
        Task Type: {task_type.value}

        Available Medical Data:
        {json.dumps(data, indent=2, default=str)}

        Please provide:
        1. A clinical analysis of the situation
        2. Possible diagnoses based on the data
        3. Recommended next steps or tests
        4. Treatment recommendations
        5. Any red flags or urgent concerns
        6. A summary for the healthcare provider

        Your response should be professional, evidence-based, and actionable.
        Always include a disclaimer that this is AI-assisted decision support and not a substitute for professional medical judgment.
        """

    def _extract_recommendations(self, reasoning: str) -> List[str]:
        """Extract recommendations from the reasoning text."""
        recommendations = []
        lines = reasoning.split('\n')
        for line in lines:
            if any(keyword in line.lower() for keyword in ['recommend', 'advise', 'suggest', 'should']):
                cleaned = line.strip().lstrip('•*-+0123456789. ')
                if len(cleaned) > 10 and cleaned not in recommendations:
                    recommendations.append(cleaned[:200])
        return recommendations[:5]  # Return top 5

    def handle_query(self, query: str) -> str:
        """
        Simplified method to handle a query and return a readable response.
        """
        response = self.process_query(query)

        output = []
        output.append("=" * 60)
        output.append("🏥 Medical Diagnostics Response")
        output.append("=" * 60)
        output.append(f"📋 Task: {response.task_type.value}")
        output.append(f"✅ Action: {response.action_taken}")
        output.append(f"🔒 Confidence: {response.confidence:.2%}")
        if response.diagnosis:
            output.append(f"🔍 Diagnosis: {response.diagnosis}")
        output.append(f"🚨 Urgency: {response.urgency}")
        if response.referral_needed:
            output.append("⚠️ Referral Needed: YES")
        output.append("-" * 60)
        output.append("📊 Clinical Analysis:")
        output.append(response.reasoning)
        if response.recommendations:
            output.append("-" * 60)
            output.append("💡 Recommendations:")
            for i, rec in enumerate(response.recommendations, 1):
                output.append(f"   {i}. {rec}")
        output.append("=" * 60)
        output.append("⚠️ DISCLAIMER: This is AI-assisted decision support.")
        output.append("   Always consult with a qualified healthcare professional.")
        output.append("   This system does not replace clinical judgment.")
        output.append("=" * 60)

        return "\n".join(output)


# ============================================================
# PART 4: DEMONSTRATION - ALL 5 QUERIES
# ============================================================

def run_medical_demo():
    """Demonstrate the Medical Diagnostics Agent in action."""
    print("\n" + "="*60)
    print("🏥  MEDICAL DIAGNOSTICS AGENT")
    print("   Complete Agentic Solution for Clinical Decision Support")
    print("="*60 + "\n")

    agent = MedicalDiagnosticsAgent()

    # ✅ FIXED: ALL 5 test queries (already correct)
    test_queries = [
        "Diagnose patient P001",
        "What are the lab results for P003?",
        "What medications is patient P002 taking?",
        "Assess risk factors for P003",
        "Triage patient P005",
    ]

    # ✅ FIXED: Run ALL 5 queries
    for i, query in enumerate(test_queries[:5], 1):
        print(f"\n{'='*60}")
        print(f"TEST {i}")
        print('='*60)
        result = agent.handle_query(query)
        print(result)


# ============================================================
# PART 5: INTERACTIVE MEDICAL CHAT
# ============================================================

def interactive_medical():
    """Interactive chat with the Medical Diagnostics Agent."""
    print("\n" + "="*60)
    print("🏥  MEDICAL AGENT - Interactive Clinical Decision Support")
    print("="*60)
    print("Type your query below. Type 'exit' to quit.")
    print("")
    print("📚 Sample Queries:")
    print("   • Diagnose patient P001")
    print("   • What are the lab results for P003?")
    print("   • What medications is patient P002 taking?")
    print("   • Assess risk factors for P003")
    print("   • Triage patient P005")
    print("   • What patients have symptoms of fever?")
    print("-"*60 + "\n")
    print("⚠️ DISCLAIMER: This is a clinical decision support tool only.")
    print("   Always consult with a qualified healthcare professional.")
    print("   This system does not replace clinical judgment.")
    print("-"*60 + "\n")

    agent = MedicalDiagnosticsAgent()

    while True:
        try:
            query = input("\n🏥 You: ").strip()
            if query.lower() in ['exit', 'quit', 'q']:
                print("\n👋 Medical Agent signing off!")
                break

            if not query:
                continue

            result = agent.handle_query(query)
            print(result)

        except KeyboardInterrupt:
            print("\n\n👋 Medical Agent signing off!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")


# ============================================================
# PART 6: CONFIGURATION
# ============================================================

config = {
    'gemma_model': 'frankmorales2020/gemma-4-e4b-stl10-topo-2026',
    'gemma_base': 'frankmorales2020/gemma-4-e4b-unesco-optimized',
    'inkling_model': INKLING_MODEL_ID,
    'inkling_max_tokens': INKLING_MAX_TOKENS,
    'inkling_temperature': INKLING_TEMPERATURE,
    'api_key_configured': bool(OPENROUTER_API_KEY),
    'system': 'Medical Diagnostics Agent',
    'version': '1.0.0',
    'patients': 5,
    'conditions': 5
}

print("\n📦 Configuration:")
print(json.dumps(config, indent=2))

# ============================================================
# RUN THE MEDICAL DEMO
# ============================================================

# ✅ RUN THE DEMO
run_medical_demo()

# Uncomment to use interactive chat instead:
# interactive_medical()

✅ API key loaded from Colab secrets! Ending: ****f808

📦 Configuration:
{
  "gemma_model": "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
  "gemma_base": "frankmorales2020/gemma-4-e4b-unesco-optimized",
  "inkling_model": "thinkingmachines/inkling",
  "inkling_max_tokens": 2048,
  "inkling_temperature": 0.7,
  "api_key_configured": true,
  "system": "Medical Diagnostics Agent",
  "version": "1.0.0",
  "patients": 5,
  "conditions": 5
}

🏥  MEDICAL DIAGNOSTICS AGENT
   Complete Agentic Solution for Clinical Decision Support


🏥  MEDICAL DIAGNOSTICS AGENT

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...
   ✅ Tokenizer loaded. Vocab size: 262144

👁️ Loading vision model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...
   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Inkling client initialized
   Model: thinkingmachines/inkling
   Max Tokens: 2048
   Temperature: 0.7
✅ Medical Diagnostics Agent Ready!
   🏥 Patients: 5
   📚 Conditions: 5


TEST 1

🏥 Medical Query: Diagnose patient P001
--------------------------------------------------
📋 Task Identification: diagnosis (confidence: 90.00%)
🧠 Inkling Reasoning: **AI-Assisted Clinical Decision Support — Patient P001 (John Smith)**
*Disclaimer: This analysis is generated by an AI assistant for decision support only. It does not replace clinical judgment, physi...
🏥 Medical Diagnostics Response
📋 Task: diagnosis
✅ Action: Retrieved patient P001
🔒 Confidence: 90.00%
🔍 Diagnosis: Further analysis needed
🚨 Urgen

## 🏥 Ferrari AI - Medical Diagnostics Agent - KIMI-K3

In [1]:
# ============================================================
# FERRARI AI - MEDICAL DIAGNOSTICS AGENT (Gemma-4 & Kimi-K3)
# Complete Agentic Solution for Clinical Decision Support
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
import random
import re

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if OPENROUTER_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{OPENROUTER_API_KEY[-4:]}")
    else:
        print("⚠️ OPENROUTER_API_KEY not found in Colab secrets.")
        OPENROUTER_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    OPENROUTER_API_KEY = ""

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ============================================================
# KIMI-K3 CONFIGURATION
# ============================================================
KIMI_MODEL_ID = "moonshotai/kimi-k3"
KIMI_MAX_TOKENS = 4096
KIMI_TEMPERATURE = 0.7

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ===========================================================
# PART 2: KIMI-K3 CLIENT
# ===========================================================

@dataclass
class KimiResponse:
    content: str
    reasoning: Optional[str] = None
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class KimiClient:
    def __init__(self,
                 api_key: Optional[str] = None,
                 model: str = KIMI_MODEL_ID,
                 max_tokens: int = KIMI_MAX_TOKENS,
                 temperature: float = KIMI_TEMPERATURE):
        self.api_key = api_key or OPENROUTER_API_KEY
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.base_url = "https://openrouter.ai/api/v1"

        if not self.api_key:
            print("⚠️ No API key found. Kimi-K3 will not work.")
        else:
            print(f"✅ Kimi-K3 client initialized")
            print(f"   Model: {model}")
            print(f"   Max Tokens: {max_tokens}")
            print(f"   Temperature: {temperature}")

        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def query(self,
              prompt: str,
              temperature: Optional[float] = None,
              max_tokens: Optional[int] = None) -> KimiResponse:
        if not self.api_key:
            return KimiResponse(
                content="ERROR: No API key. Please add OPENROUTER_API_KEY to Colab secrets."
            )

        temp = temperature if temperature is not None else self.temperature
        max_tok = max_tokens if max_tokens is not None else self.max_tokens

        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temp,
            "max_tokens": max_tok,
            "reasoning": {"enabled": True}
        }

        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60
            )

            if response.status_code != 200:
                error_content = f"API Error ({response.status_code}): {response.text[:200]}"
                return KimiResponse(content=error_content)

            data = response.json()
            message = data['choices'][0]['message']

            content = message.get('content')
            reasoning = message.get('reasoning')

            if not content and reasoning:
                content = reasoning
                reasoning = None

            if not content:
                content = "No response content received."

            return KimiResponse(
                content=content,
                reasoning=reasoning,
                raw_response=data,
                model=data.get('model')
            )

        except Exception as e:
            return KimiResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: MEDICAL DIAGNOSTICS AGENT
# ============================================================

class MedicalTaskType(Enum):
    SYMPTOM_CHECK = "symptom_check"
    DIAGNOSIS = "diagnosis"
    MEDICATION_CHECK = "medication_check"
    LAB_INTERPRETATION = "lab_interpretation"
    RISK_ASSESSMENT = "risk_assessment"
    TREATMENT_RECOMMENDATION = "treatment_recommendation"
    TRIAGE = "triage"
    GENERAL = "general"


@dataclass
class Patient:
    patient_id: str
    name: str
    age: int
    gender: str
    symptoms: List[str]
    existing_conditions: List[str]
    medications: List[str]
    allergies: List[str]
    vital_signs: Dict[str, float]
    lab_results: Dict[str, Any]
    visit_date: datetime

    def to_dict(self) -> Dict:
        return {
            'patient_id': self.patient_id,
            'name': self.name,
            'age': self.age,
            'gender': self.gender,
            'symptoms': self.symptoms,
            'existing_conditions': self.existing_conditions,
            'medications': self.medications,
            'allergies': self.allergies,
            'vital_signs': self.vital_signs,
            'lab_results': self.lab_results,
            'visit_date': self.visit_date.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class MedicalKnowledge:
    condition: str
    description: str
    common_symptoms: List[str]
    risk_factors: List[str]
    treatment_options: List[str]
    urgency_level: str


@dataclass
class MedicalResponse:
    task_type: MedicalTaskType
    action_taken: str
    data: Dict
    reasoning: str
    diagnosis: Optional[str] = None
    confidence: float = 0.0
    recommendations: List[str] = field(default_factory=list)
    referral_needed: bool = False
    urgency: str = "LOW"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class MedicalDiagnosticsAgent:
    """
    Agentic AI System for Medical Diagnostics and Clinical Decision Support
    Combines Gemma-4 classification with Kimi-K3 reasoning
    """

    def __init__(self):
        print("\n" + "="*60)
        print("🏥  MEDICAL DIAGNOSTICS AGENT (Gemma-4 & Kimi-K3)")
        print("="*60)

        # Initialize components
        self.gemma = GemmaTOPOCertified()
        self.kimi = KimiClient()

        self.patients: Dict[str, Patient] = {}
        self.medical_knowledge: Dict[str, MedicalKnowledge] = {}
        self._initialize_patient_data()
        self._initialize_medical_knowledge()

        print("="*60)
        print("✅ Medical Diagnostics Agent Ready!")
        print(f"   🏥 Patients: {len(self.patients)}")
        print(f"   📚 Conditions: {len(self.medical_knowledge)}")
        print("="*60 + "\n")

    def _initialize_patient_data(self):
        now = datetime.now()
        patients_data = [
            (
                "P001", "John Smith", 45, "Male",
                ["fever", "cough", "shortness of breath"],
                ["hypertension", "type 2 diabetes"],
                ["lisinopril", "metformin"],
                ["penicillin"],
                {"bp": "140/90", "hr": 95, "temp": 38.5, "spo2": 94},
                {"wbc": "12.5", "crp": "15.2", "glucose": "140"},
                now - timedelta(hours=2)
            ),
            (
                "P002", "Sarah Johnson", 32, "Female",
                ["headache", "nausea", "photophobia"],
                ["migraine history"],
                ["sumatriptan"],
                ["sulfa drugs"],
                {"bp": "120/80", "hr": 72, "temp": 37.0, "spo2": 99},
                {"wbc": "7.2", "crp": "1.5", "glucose": "95"},
                now - timedelta(days=1)
            ),
            (
                "P003", "Robert Chen", 65, "Male",
                ["chest pain", "dizziness", "fatigue"],
                ["coronary artery disease", "hyperlipidemia"],
                ["atorvastatin", "aspirin", "metoprolol"],
                [],
                {"bp": "150/95", "hr": 88, "temp": 37.2, "spo2": 97},
                {"wbc": "9.8", "crp": "8.7", "glucose": "120", "troponin": "0.08"},
                now - timedelta(hours=4)
            ),
            (
                "P004", "Maria Garcia", 28, "Female",
                ["abdominal pain", "nausea", "vomiting"],
                ["GERD"],
                ["omeprazole"],
                ["latex"],
                {"bp": "115/75", "hr": 80, "temp": 37.5, "spo2": 98},
                {"wbc": "11.2", "crp": "5.8", "glucose": "90"},
                now - timedelta(hours=6)
            ),
            (
                "P005", "James Wilson", 55, "Male",
                ["fever", "chills", "muscle aches", "fatigue"],
                ["none"],
                [],
                [],
                {"bp": "135/85", "hr": 100, "temp": 39.0, "spo2": 96},
                {"wbc": "14.5", "crp": "22.3", "glucose": "110"},
                now - timedelta(hours=1)
            ),
        ]

        for patient_data in patients_data:
            patient = Patient(
                patient_id=patient_data[0],
                name=patient_data[1],
                age=patient_data[2],
                gender=patient_data[3],
                symptoms=patient_data[4],
                existing_conditions=patient_data[5],
                medications=patient_data[6],
                allergies=patient_data[7],
                vital_signs=patient_data[8],
                lab_results=patient_data[9],
                visit_date=patient_data[10]
            )
            self.patients[patient.patient_id] = patient

    def _initialize_medical_knowledge(self):
        knowledge_data = [
            MedicalKnowledge(
                condition="Pneumonia",
                description="Inflammation of the lungs typically caused by infection",
                common_symptoms=["fever", "cough", "shortness of breath", "chest pain"],
                risk_factors=["age > 65", "smoking", "immunosuppression", "chronic lung disease"],
                treatment_options=["Antibiotics", "Oxygen therapy", "Cough medicine"],
                urgency_level="HIGH"
            ),
            MedicalKnowledge(
                condition="Migraine",
                description="Recurrent headaches often with nausea and sensitivity to light",
                common_symptoms=["headache", "nausea", "photophobia", "aura"],
                risk_factors=["family history", "female gender", "stress"],
                treatment_options=["Pain relievers", "Triptans", "Preventive medications"],
                urgency_level="LOW"
            ),
            MedicalKnowledge(
                condition="Myocardial Infarction",
                description="Heart attack due to reduced blood flow to the heart",
                common_symptoms=["chest pain", "shortness of breath", "dizziness", "nausea"],
                risk_factors=["age", "hypertension", "high cholesterol", "smoking"],
                treatment_options=["Aspirin", "Nitroglycerin", "PCI", "Thrombolytics"],
                urgency_level="CRITICAL"
            ),
            MedicalKnowledge(
                condition="Gastroenteritis",
                description="Inflammation of the stomach and intestines",
                common_symptoms=["abdominal pain", "nausea", "vomiting", "diarrhea"],
                risk_factors=["Contaminated food/water", "Poor hygiene"],
                treatment_options=["Fluid replacement", "Anti-emetics", "Antidiarrheals"],
                urgency_level="LOW"
            ),
            MedicalKnowledge(
                condition="Sepsis",
                description="Life-threatening organ dysfunction caused by infection",
                common_symptoms=["fever", "chills", "confusion", "rapid heart rate"],
                risk_factors=["Elderly", "Immunocompromised", "Chronic illness"],
                treatment_options=["Antibiotics", "IV fluids", "Vasopressors"],
                urgency_level="CRITICAL"
            ),
        ]

        for knowledge in knowledge_data:
            self.medical_knowledge[knowledge.condition] = knowledge

    def identify_task(self, query: str) -> Dict:
        query_lower = query.lower()

        task_mapping = {
            'symptom': MedicalTaskType.SYMPTOM_CHECK,
            'symptoms': MedicalTaskType.SYMPTOM_CHECK,
            'diagnose': MedicalTaskType.DIAGNOSIS,
            'diagnosis': MedicalTaskType.DIAGNOSIS,
            'medication': MedicalTaskType.MEDICATION_CHECK,
            'drug': MedicalTaskType.MEDICATION_CHECK,
            'lab': MedicalTaskType.LAB_INTERPRETATION,
            'test': MedicalTaskType.LAB_INTERPRETATION,
            'result': MedicalTaskType.LAB_INTERPRETATION,
            'risk': MedicalTaskType.RISK_ASSESSMENT,
            'treatment': MedicalTaskType.TREATMENT_RECOMMENDATION,
            'urgent': MedicalTaskType.TRIAGE,
            'emergency': MedicalTaskType.TRIAGE,
            'triage': MedicalTaskType.TRIAGE,
        }

        for keyword, task_type in task_mapping.items():
            if keyword in query_lower:
                return {'task_type': task_type, 'confidence': 0.9}

        return {'task_type': MedicalTaskType.GENERAL, 'confidence': 0.6}

    def _get_patient(self, patient_id: str) -> Optional[Dict]:
        if patient_id in self.patients:
            return self.patients[patient_id].to_dict()
        return None

    def _get_condition_knowledge(self, condition: str) -> Optional[Dict]:
        if condition in self.medical_knowledge:
            return self.medical_knowledge[condition].__dict__
        return None

    def process_query(self, query: str) -> MedicalResponse:
        print(f"\n🏥 Medical Query: {query}")
        print("-" * 50)

        task_decision = self.identify_task(query)
        task_type = task_decision['task_type']
        confidence = task_decision['confidence']

        print(f"📋 Task Identification: {task_type.value} (confidence: {confidence:.2%})")

        # ✅ FIXED: Correct regex pattern for patient ID extraction
        patient_match = re.search(r'P\d{3}', query.upper())
        patient_id = patient_match.group(0) if patient_match else None

        conditions = list(self.medical_knowledge.keys())
        mentioned_condition = None
        for condition in conditions:
            if condition.lower() in query.lower():
                mentioned_condition = condition
                break

        data = {}
        action_taken = ""
        diagnosis = None
        referral_needed = False
        urgency = "LOW"

        if task_type == MedicalTaskType.DIAGNOSIS or task_type == MedicalTaskType.SYMPTOM_CHECK:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    action_taken = f"Retrieved patient {patient_id}"

                    matching_conditions = []
                    for condition, knowledge in self.medical_knowledge.items():
                        if any(symptom.lower() in condition.lower() for symptom in patient_data['symptoms']):
                            matching_conditions.append(condition)
                        elif any(symptom.lower() in ' '.join(knowledge.common_symptoms).lower()
                                for symptom in patient_data['symptoms']):
                            matching_conditions.append(condition)

                    if matching_conditions:
                        data['possible_conditions'] = matching_conditions
                        data['condition_details'] = [
                            self._get_condition_knowledge(c) for c in matching_conditions[:3]
                        ]
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['all_patients'] = [p.to_dict() for p in self.patients.values()]
                action_taken = "Retrieved all patients"

        elif task_type == MedicalTaskType.LAB_INTERPRETATION:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    data['lab_results'] = patient_data.get('lab_results', {})
                    action_taken = f"Retrieved lab results for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['lab_results'] = []
                for patient in self.patients.values():
                    data['lab_results'].append({
                        'patient_id': patient.patient_id,
                        'name': patient.name,
                        'results': patient.lab_results
                    })
                action_taken = "Retrieved lab results for all patients"

        elif task_type == MedicalTaskType.MEDICATION_CHECK:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    data['medications'] = patient_data.get('medications', [])
                    data['allergies'] = patient_data.get('allergies', [])
                    action_taken = f"Retrieved medication information for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['medications_summary'] = {}
                for patient in self.patients.values():
                    data['medications_summary'][patient.patient_id] = {
                        'name': patient.name,
                        'medications': patient.medications,
                        'allergies': patient.allergies
                    }
                action_taken = "Retrieved medication information for all patients"

        elif task_type == MedicalTaskType.RISK_ASSESSMENT:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    risk_factors = []
                    if patient_data['age'] > 65:
                        risk_factors.append("Age > 65")
                    if patient_data.get('existing_conditions'):
                        risk_factors.append("Existing conditions")
                    if patient_data.get('vital_signs', {}).get('bp'):
                        bp = patient_data['vital_signs']['bp']
                        if bp != "Normal":
                            risk_factors.append(f"Abnormal blood pressure: {bp}")
                    data['risk_factors'] = risk_factors
                    action_taken = f"Assessed risk factors for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['risk_assessment'] = "Please specify a patient ID"
                action_taken = "Risk assessment requires patient ID"

        elif task_type == MedicalTaskType.TRIAGE:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    symptoms = patient_data.get('symptoms', [])

                    if any(s in ['chest pain', 'shortness of breath'] for s in symptoms):
                        urgency = "CRITICAL"
                        referral_needed = True
                    elif 'fever' in symptoms and any(t in symptoms for t in ['chills', 'muscle aches']):
                        urgency = "HIGH"
                    elif 'headache' in symptoms:
                        urgency = "MEDIUM"
                    else:
                        urgency = "LOW"

                    data['triage_level'] = urgency
                    action_taken = f"Triaged patient {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['triage_summary'] = [
                    {'patient_id': p.patient_id, 'name': p.name, 'symptoms': p.symptoms}
                    for p in self.patients.values()
                ]
                action_taken = "Retrieved triage summary for all patients"

        else:
            data['message'] = "General medical inquiry received"
            action_taken = "General query processed"

        # Generate reasoning using Kimi-K3
        reasoning_prompt = self._generate_reasoning_prompt(query, task_type, data)
        kimi_response = self.kimi.query(reasoning_prompt)

        print(f"🧠 Kimi-K3 Reasoning: {kimi_response.content[:200]}...")

        return MedicalResponse(
            task_type=task_type,
            action_taken=action_taken,
            data=data,
            reasoning=kimi_response.content,
            diagnosis=mentioned_condition or "Further analysis needed",
            confidence=confidence,
            recommendations=self._extract_recommendations(kimi_response.content),
            referral_needed=referral_needed,
            urgency=urgency,
            timestamp=datetime.now().isoformat()
        )

    def _generate_reasoning_prompt(self, query: str, task_type: MedicalTaskType, data: Dict) -> str:
        return f"""
        You are a Medical Diagnostics AI Assistant providing clinical decision support.

        User Query: "{query}"
        Task Type: {task_type.value}

        Available Medical Data:
        {json.dumps(data, indent=2, default=str)}

        Please provide:
        1. A clinical analysis of the situation
        2. Possible diagnoses based on the data
        3. Recommended next steps or tests
        4. Treatment recommendations
        5. Any red flags or urgent concerns
        6. A summary for the healthcare provider

        Your response should be professional, evidence-based, and actionable.
        Always include a disclaimer that this is AI-assisted decision support and not a substitute for professional medical judgment.
        """

    def _extract_recommendations(self, reasoning: str) -> List[str]:
        recommendations = []
        lines = reasoning.split('\n')
        for line in lines:
            if any(keyword in line.lower() for keyword in ['recommend', 'advise', 'suggest', 'should']):
                cleaned = line.strip().lstrip('•*-+0123456789. ')
                if len(cleaned) > 10 and cleaned not in recommendations:
                    recommendations.append(cleaned[:200])
        return recommendations[:5]

    def handle_query(self, query: str) -> str:
        response = self.process_query(query)

        output = []
        output.append("=" * 60)
        output.append("🏥 Medical Diagnostics Response")
        output.append("=" * 60)
        output.append(f"📋 Task: {response.task_type.value}")
        output.append(f"✅ Action: {response.action_taken}")
        output.append(f"🔒 Confidence: {response.confidence:.2%}")
        if response.diagnosis:
            output.append(f"🔍 Diagnosis: {response.diagnosis}")
        output.append(f"🚨 Urgency: {response.urgency}")
        if response.referral_needed:
            output.append("⚠️ Referral Needed: YES")
        output.append("-" * 60)
        output.append("📊 Clinical Analysis:")
        output.append(response.reasoning)
        if response.recommendations:
            output.append("-" * 60)
            output.append("💡 Recommendations:")
            for i, rec in enumerate(response.recommendations, 1):
                output.append(f"   {i}. {rec}")
        output.append("=" * 60)
        output.append("⚠️ DISCLAIMER: This is AI-assisted decision support.")
        output.append("   Always consult with a qualified healthcare professional.")
        output.append("   This system does not replace clinical judgment.")
        output.append("=" * 60)

        return "\n".join(output)


# ============================================================
# PART 4: DEMONSTRATION - ALL 5 QUERIES
# ============================================================

def run_medical_demo():
    print("\n" + "="*60)
    print("🏥  MEDICAL DIAGNOSTICS AGENT (Kimi-K3 Edition)")
    print("   Complete Agentic Solution for Clinical Decision Support")
    print("="*60 + "\n")

    agent = MedicalDiagnosticsAgent()

    test_queries = [
        "Diagnose patient P001",
        "What are the lab results for P003?",
        "What medications is patient P002 taking?",
        "Assess risk factors for P003",
        "Triage patient P005",
    ]

    # ✅ FIXED: Run ALL 5 queries (removed [:2] limit)
    for i, query in enumerate(test_queries, 1):
        print(f"\n{'='*60}")
        print(f"TEST {i}")
        print('='*60)
        result = agent.handle_query(query)
        print(result)


# ============================================================
# RUN DEMO
# ============================================================
if __name__ == "__main__":
    run_medical_demo()

✅ API key loaded from Colab secrets! Ending: ****f808

🏥  MEDICAL DIAGNOSTICS AGENT (Kimi-K3 Edition)
   Complete Agentic Solution for Clinical Decision Support


🏥  MEDICAL DIAGNOSTICS AGENT (Gemma-4 & Kimi-K3)

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...
   ✅ Tokenizer loaded. Vocab size: 262144

👁️ Loading vision model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...
   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Kimi-K3 client initialized
   Model: moonshotai/kimi-k3
   Max Tokens: 4096
   Temperature: 0.7
✅ Medical Diagnostics Agent Ready!
   🏥 Patients: 5
   📚 Conditions: 5


TEST 1

🏥 Medical Query: Diagnose patient P001
--------------------------------------------------
📋 Task Identification: diagnosis (confidence: 90.00%)
🧠 Kimi-K3 Reasoning: # Clinical Decision Support Report — Patient P001

**Patient:** John Smith | 45M | Visit: 2026-08-04 15:12
**Chief Concerns:** Fever, cough, shortness of breath

---

## 1. Clinical Analysis

**Presen...
🏥 Medical Diagnostics Response
📋 Task: diagnosis
✅ Action: Retrieved patient P001
🔒 Confidence: 90.00%
🔍 Diagnosis: Further analysis needed
🚨 Urgency: LO

##  Ferrari AI - Medical Diagnostics Agent - FABLE-5

In [5]:
# ============================================================
# COMPLETE FABLE 5 CLIENT FOR MEDICAL AGENT
# No dependencies on other engines - runs standalone
# ============================================================

import anthropic
from google.colab import userdata
from typing import List, Dict, Any, Optional

# ============================================================
# 1. FABLE 5 CLIENT (STANDALONE)
# ============================================================

class ClaudeFable5Client:
    """
    Claude Fable 5 reasoning engine client.
    Fully functional, no external dependencies.
    """

    def __init__(
        self,
        api_key: Optional[str] = None,
        effort: str = "high",  # "low", "medium", "high"
        max_tokens: int = 4096
    ):
        """
        Initialize Fable 5 client.

        Args:
            api_key: Anthropic API key (or use ANTHROPIC_API_KEY secret)
            effort: Reasoning effort - "low", "medium", "high"
            max_tokens: Maximum output tokens
        """
        try:
            self.client = anthropic.Anthropic(
                api_key=api_key or userdata.get('ANTHROPIC_API_KEY')
            )
            self.model = "claude-fable-5"
            self.effort = effort
            self.max_tokens = max_tokens
            print(f"✅ Fable 5 initialized (effort={effort})")
        except Exception as e:
            print(f"❌ Failed to initialize Fable 5: {e}")
            raise

    def query(
        self,
        messages: List[Dict[str, str]],
        max_tokens: Optional[int] = None,
        effort: Optional[str] = None,
        **kwargs
    ) -> Dict[str, Any]:
        """
        Query Claude Fable 5 with adaptive thinking.

        IMPORTANT FABLE 5 BEHAVIOR:
        - temperature, top_p, top_k are IGNORED
        - Use 'effort' parameter to control reasoning depth
        - Refusals return HTTP 200 with stop_reason: "refusal"

        Args:
            messages: List of message dicts with 'role' and 'content'
            max_tokens: Override default max_tokens
            effort: Override default effort ("low", "medium", "high")

        Returns:
            Dict with 'content', 'usage', 'stop_reason', etc.
        """

        # Use specified effort or fall back to instance default
        reasoning_effort = effort or self.effort

        # Extract system prompt and format messages
        system_prompt = None
        anthropic_messages = []

        for msg in messages:
            role = msg.get("role", "")
            content = msg.get("content", "")

            if role == "system":
                system_prompt = content
            elif role in ["user", "assistant"]:
                anthropic_messages.append({
                    "role": role,
                    "content": content
                })
            elif role == "function":
                # Handle function/tool calls if needed
                anthropic_messages.append({
                    "role": "user",
                    "content": f"[Function Result: {content}]"
                })

        try:
            # Prepare the request parameters
            # CRITICAL: NO temperature, top_p, or top_k - Fable 5 ignores them
            request_params = {
                "model": self.model,
                "messages": anthropic_messages,
                "max_tokens": max_tokens or self.max_tokens,
            }

            # Add system prompt if present
            if system_prompt:
                request_params["system"] = system_prompt

            # Add reasoning effort (Fable 5 specific)
            if reasoning_effort:
                request_params["extra_headers"] = {
                    "anthropic-adaptive-thinking": reasoning_effort
                }

            # Make the API call
            response = self.client.messages.create(**request_params)

            # Extract the text content
            content_text = ""
            is_refusal = False

            for block in response.content:
                if block.type == "text":
                    content_text += block.text
                elif block.type == "refusal":
                    content_text = f"REFUSAL: {block.text}"
                    is_refusal = True

            # Check stop reason
            if response.stop_reason == "refusal":
                is_refusal = True
                if not content_text.startswith("REFUSAL:"):
                    content_text = f"REFUSAL: The model refused to respond to this request."

            # Build result
            result = {
                "content": content_text,
                "stop_reason": response.stop_reason,
                "usage": {
                    "input_tokens": response.usage.input_tokens,
                    "output_tokens": response.usage.output_tokens,
                    "total_tokens": response.usage.input_tokens + response.usage.output_tokens
                },
                "engine": "claude_fable_5",
                "effort": reasoning_effort,
                "refusal": is_refusal,
                "success": True
            }

            return result

        except Exception as e:
            error_msg = str(e)
            return {
                "content": f"Fable 5 Error: {error_msg}",
                "error": True,
                "engine": "claude_fable_5",
                "success": False,
                "error_message": error_msg
            }

# ============================================================
# 2. SIMPLE MEDICAL CLINICAL CASE
# ============================================================

def create_clinical_case():
    """Create a STEMI clinical case for testing."""

    system_prompt = """You are a clinical decision support AI with expertise in cardiology. You must:
1. Provide evidence-based assessments
2. Reference recognized clinical guidelines (ACC/AHA, ESC)
3. Structure responses with clear sections
4. Include actionable recommendations
5. Always prioritize patient safety

Guidelines to reference:
- ACC/AHA 2025 STEMI Guidelines
- ESC Guidelines for Acute Coronary Syndromes"""

    clinical_case = """
# CLINICAL CASE - ACUTE CHEST PAIN

## Patient Demographics
- **Age:** 67 years
- **Sex:** Male
- **BMI:** 31.0

## Comorbidities
- Hypertension
- Type 2 Diabetes Mellitus
- Hyperlipidemia

## Presenting Symptoms
- Sudden onset chest pain (10/10 severity)
- Radiating to left arm
- Diaphoresis (profuse sweating)
- Shortness of breath
- Nausea

## Vital Signs
- Heart Rate: 98 bpm
- Blood Pressure: 145/90 mmHg
- Respiratory Rate: 22/min
- Oxygen Saturation: 94% on room air
- Temperature: 37.2°C

## Laboratory Results
- Troponin I: 0.08 ng/mL (Reference: <0.04 ng/mL) - **ELEVATED**
- CK-MB: 25 U/L (Reference: <5 U/L) - **ELEVATED**
- BNP: 180 pg/mL (Reference: <100 pg/mL) - **ELEVATED**
- Glucose: 168 mg/dL (Reference: 70-100 mg/dL)
- Creatinine: 1.2 mg/dL (Reference: 0.6-1.2 mg/dL)

## ECG Findings
- Sinus tachycardia
- ST elevation in leads V2-V4
- ST depression in leads III and aVF (reciprocal changes)
- T wave inversions

## Clinical History
Patient with history of hypertension (on losartan) and diabetes (on metformin). Last seen at clinic 2 months ago with normal ECG. No known coronary artery disease.

---
**Please provide:**
1. Differential diagnosis
2. Most likely diagnosis with supporting evidence
3. Immediate management plan
4. Recommended diagnostic workup
5. Treatment recommendations with guideline references
6. Disposition recommendation
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": clinical_case}
    ]

    return messages

# ============================================================
# 3. TEST FABLE 5
# ============================================================

def test_fable5():
    """Test Fable 5 with a clinical case."""

    print("="*70)
    print("🏥 FABLE 5 - CLINICAL REASONING TEST")
    print("="*70)

    # Initialize Fable 5
    print("\n🔄 Initializing Fable 5...")

    try:
        client = ClaudeFable5Client(effort="high")
    except Exception as e:
        print(f"\n❌ Could not initialize Fable 5: {e}")
        print("\n💡 Make sure you have:")
        print("   1. ANTHROPIC_API_KEY set in Colab secrets")
        print("   2. Access to claude-fable-5 model")
        return

    # Create clinical case
    print("\n📋 Preparing clinical case...")
    messages = create_clinical_case()

    # Query Fable 5
    print("\n🧠 Fable 5 is reasoning... (this may take 10-20 seconds)")
    print("-"*70)

    result = client.query(messages)

    # Display results
    print("\n" + "="*70)
    print("📊 FABLE 5 CLINICAL REASONING RESULTS")
    print("="*70)

    if result.get("success", False):
        # Check for refusal
        if result.get("refusal", False):
            print("\n🚫 THE MODEL REFUSED THIS REQUEST")
            print("   This may happen for sensitive clinical questions.")
            print("   Try adjusting the prompt or using a different effort level.")
            print("\n" + "-"*70)
            print(result["content"])
        else:
            # Print the response
            print("\n" + result["content"])

            # Print usage statistics
            if "usage" in result:
                usage = result["usage"]
                print("\n" + "="*70)
                print("📊 USAGE STATISTICS")
                print("="*70)
                print(f"  Input Tokens:  {usage['input_tokens']:,}")
                print(f"  Output Tokens: {usage['output_tokens']:,}")
                print(f"  Total Tokens:  {usage['total_tokens']:,}")
                print(f"  Effort Level:  {result.get('effort', 'N/A')}")
                print(f"  Stop Reason:   {result.get('stop_reason', 'N/A')}")
    else:
        print("\n❌ ERROR OCCURRED")
        print(f"   {result.get('content', 'Unknown error')}")
        print(f"   Error: {result.get('error_message', 'No details')}")

    return result

# ============================================================
# 4. COMPARE EFFORT LEVELS
# ============================================================

def compare_effort_levels():
    """Compare Fable 5 with different effort levels."""

    print("="*70)
    print("🔄 COMPARING FABLE 5 EFFORT LEVELS")
    print("="*70)
    print("\nNOTE: Higher effort = better reasoning but slower/more expensive")
    print("-"*70)

    # Test case
    messages = [
        {"role": "user", "content": "What is the differential diagnosis for a 67-year-old male with chest pain, elevated troponin, and ST elevation on ECG?"}
    ]

    results = {}

    for effort in ["low", "medium", "high"]:
        print(f"\n📊 EFFORT: {effort.upper()}")
        print("-"*40)

        try:
            client = ClaudeFable5Client(effort=effort, max_tokens=500)
            result = client.query(messages)

            if result.get("success", False):
                content = result.get("content", "")
                if len(content) > 200:
                    content = content[:200] + "..."
                print(f"Response: {content}")
                if "usage" in result:
                    total = result["usage"].get("total_tokens", 0)
                    print(f"Tokens: {total}")
                results[effort] = result
            else:
                print(f"❌ Error: {result.get('content', 'Unknown')}")

        except Exception as e:
            print(f"❌ Failed: {str(e)}")

    print("\n" + "="*70)
    print("✅ Comparison complete")
    print("="*70)

# ============================================================
# 5. MAIN EXECUTION
# ============================================================

def main():
    """Main execution function."""

    print("="*70)
    print("🏥 CLAUDE FABLE 5 - CLINICAL REASONING ENGINE")
    print("   CF-Free Compatible | Adaptive Thinking")
    print("="*70)

    print("\n📋 IMPORTANT FABLE 5 NOTES:")
    print("   • Adaptive thinking is ALWAYS ON (cannot disable)")
    print("   • temperature, top_p, top_k are IGNORED")
    print("   • Use 'effort' to control reasoning depth")
    print("   • Refusals are returned gracefully")
    print("   • Higher effort = better reasoning, more tokens, slower")
    print("="*70)

    # Run the test
    test_fable5()

# ============================================================
# 6. RUN THE CODE
# ============================================================

if __name__ == "__main__":
    main()

🏥 CLAUDE FABLE 5 - CLINICAL REASONING ENGINE
   CF-Free Compatible | Adaptive Thinking

📋 IMPORTANT FABLE 5 NOTES:
   • Adaptive thinking is ALWAYS ON (cannot disable)
   • temperature, top_p, top_k are IGNORED
   • Use 'effort' to control reasoning depth
   • Refusals are returned gracefully
   • Higher effort = better reasoning, more tokens, slower
🏥 FABLE 5 - CLINICAL REASONING TEST

🔄 Initializing Fable 5...
✅ Fable 5 initialized (effort=high)

📋 Preparing clinical case...

🧠 Fable 5 is reasoning... (this may take 10-20 seconds)
----------------------------------------------------------------------

📊 FABLE 5 CLINICAL REASONING RESULTS

# CLINICAL DECISION SUPPORT — ACUTE CHEST PAIN

> ⚠️ **CRITICAL ALERT: This presentation meets criteria for STEMI. Time-sensitive emergency. Activate cardiac catheterization lab immediately.**

---

## 1. Differential Diagnosis

| Diagnosis | Likelihood | Rationale |
|---|---|---|
| **Anterior STEMI (LAD territory)** | **Very High** | ST elevation V

GEMMA4-FABLE5

In [1]:
# ============================================================
# FERRARI AI - MEDICAL DIAGNOSTICS AGENT (Gemma-4 & Fable-5)
# Complete Agentic Solution for Clinical Decision Support
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
import random
import re

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    if ANTHROPIC_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{ANTHROPIC_API_KEY[-4:]}")
    else:
        print("⚠️ ANTHROPIC_API_KEY not found in Colab secrets.")
        ANTHROPIC_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    ANTHROPIC_API_KEY = ""

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# ============================================================
# FABLE-5 CONFIGURATION
# ============================================================
FABLE_MODEL = "claude-fable-5"
FABLE_MAX_TOKENS = 4096

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ===========================================================
# PART 2: FABLE-5 CLIENT (FIXED - Handles Thinking Blocks)
# ===========================================================

@dataclass
class FableResponse:
    content: str
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class FableClient:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key or ANTHROPIC_API_KEY

        if not self.api_key:
            print("⚠️ No API key found. Fable-5 will not work.")
            self.client = None
        else:
            try:
                import anthropic
                self.client = anthropic.Anthropic(api_key=self.api_key)
                print(f"✅ Fable-5 client initialized")
                print(f"   Model: {FABLE_MODEL}")
            except Exception as e:
                print(f"❌ Failed to initialize Fable-5: {e}")
                self.client = None

    def query(self, prompt: str) -> FableResponse:
        if not self.client:
            return FableResponse(
                content="ERROR: No API key. Please add ANTHROPIC_API_KEY to Colab secrets."
            )

        try:
            response = self.client.messages.create(
                model=FABLE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=FABLE_MAX_TOKENS,
                extra_headers={
                    "anthropic-adaptive-thinking": "high"
                }
            )

            # Handle all block types (text, thinking, redacted_thinking)
            content_parts = []
            for block in response.content:
                if block.type == "text":
                    content_parts.append(block.text)
                elif block.type == "thinking":
                    # Fable-5 internal reasoning
                    if hasattr(block, 'thinking'):
                        content_parts.append(block.thinking)
                    elif hasattr(block, 'text'):
                        content_parts.append(block.text)
                    else:
                        content_parts.append(str(block))
                elif block.type == "redacted_thinking":
                    # Skip redacted thinking to keep response clean
                    pass
                else:
                    # Fallback for unknown block types
                    if hasattr(block, 'text'):
                        content_parts.append(block.text)
                    else:
                        content_parts.append(str(block))

            # If no text content, try to get thinking content
            if not content_parts:
                content_parts = ["No content received from Fable-5."]

            full_content = "\n".join(content_parts)

            return FableResponse(
                content=full_content,
                raw_response=response.__dict__,
                model=FABLE_MODEL
            )

        except Exception as e:
            return FableResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: MEDICAL DIAGNOSTICS AGENT
# ============================================================

class MedicalTaskType(Enum):
    SYMPTOM_CHECK = "symptom_check"
    DIAGNOSIS = "diagnosis"
    MEDICATION_CHECK = "medication_check"
    LAB_INTERPRETATION = "lab_interpretation"
    RISK_ASSESSMENT = "risk_assessment"
    TREATMENT_RECOMMENDATION = "treatment_recommendation"
    TRIAGE = "triage"
    GENERAL = "general"


@dataclass
class Patient:
    patient_id: str
    name: str
    age: int
    gender: str
    symptoms: List[str]
    existing_conditions: List[str]
    medications: List[str]
    allergies: List[str]
    vital_signs: Dict[str, float]
    lab_results: Dict[str, Any]
    visit_date: datetime

    def to_dict(self) -> Dict:
        return {
            'patient_id': self.patient_id,
            'name': self.name,
            'age': self.age,
            'gender': self.gender,
            'symptoms': self.symptoms,
            'existing_conditions': self.existing_conditions,
            'medications': self.medications,
            'allergies': self.allergies,
            'vital_signs': self.vital_signs,
            'lab_results': self.lab_results,
            'visit_date': self.visit_date.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class MedicalKnowledge:
    condition: str
    description: str
    common_symptoms: List[str]
    risk_factors: List[str]
    treatment_options: List[str]
    urgency_level: str


@dataclass
class MedicalResponse:
    task_type: MedicalTaskType
    action_taken: str
    data: Dict
    reasoning: str
    diagnosis: Optional[str] = None
    confidence: float = 0.0
    recommendations: List[str] = field(default_factory=list)
    referral_needed: bool = False
    urgency: str = "LOW"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class MedicalDiagnosticsAgent:
    """
    Agentic AI System for Medical Diagnostics and Clinical Decision Support
    Combines Gemma-4 classification with Fable-5 reasoning
    """

    def __init__(self):
        print("\n" + "="*60)
        print("🏥  MEDICAL DIAGNOSTICS AGENT (Gemma-4 & Fable-5)")
        print("="*60)

        # Initialize components
        self.gemma = GemmaTOPOCertified()
        self.fable = FableClient()

        self.patients: Dict[str, Patient] = {}
        self.medical_knowledge: Dict[str, MedicalKnowledge] = {}
        self._initialize_patient_data()
        self._initialize_medical_knowledge()

        print("="*60)
        print("✅ Medical Diagnostics Agent Ready!")
        print(f"   🏥 Patients: {len(self.patients)}")
        print(f"   📚 Conditions: {len(self.medical_knowledge)}")
        print("="*60 + "\n")

    def _initialize_patient_data(self):
        now = datetime.now()
        patients_data = [
            (
                "P001", "John Smith", 45, "Male",
                ["fever", "cough", "shortness of breath"],
                ["hypertension", "type 2 diabetes"],
                ["lisinopril", "metformin"],
                ["penicillin"],
                {"bp": "140/90", "hr": 95, "temp": 38.5, "spo2": 94},
                {"wbc": "12.5", "crp": "15.2", "glucose": "140"},
                now - timedelta(hours=2)
            ),
            (
                "P002", "Sarah Johnson", 32, "Female",
                ["headache", "nausea", "photophobia"],
                ["migraine history"],
                ["sumatriptan"],
                ["sulfa drugs"],
                {"bp": "120/80", "hr": 72, "temp": 37.0, "spo2": 99},
                {"wbc": "7.2", "crp": "1.5", "glucose": "95"},
                now - timedelta(days=1)
            ),
            (
                "P003", "Robert Chen", 65, "Male",
                ["chest pain", "dizziness", "fatigue"],
                ["coronary artery disease", "hyperlipidemia"],
                ["atorvastatin", "aspirin", "metoprolol"],
                [],
                {"bp": "150/95", "hr": 88, "temp": 37.2, "spo2": 97},
                {"wbc": "9.8", "crp": "8.7", "glucose": "120", "troponin": "0.08"},
                now - timedelta(hours=4)
            ),
            (
                "P004", "Maria Garcia", 28, "Female",
                ["abdominal pain", "nausea", "vomiting"],
                ["GERD"],
                ["omeprazole"],
                ["latex"],
                {"bp": "115/75", "hr": 80, "temp": 37.5, "spo2": 98},
                {"wbc": "11.2", "crp": "5.8", "glucose": "90"},
                now - timedelta(hours=6)
            ),
            (
                "P005", "James Wilson", 55, "Male",
                ["fever", "chills", "muscle aches", "fatigue"],
                ["none"],
                [],
                [],
                {"bp": "135/85", "hr": 100, "temp": 39.0, "spo2": 96},
                {"wbc": "14.5", "crp": "22.3", "glucose": "110"},
                now - timedelta(hours=1)
            ),
        ]

        for patient_data in patients_data:
            patient = Patient(
                patient_id=patient_data[0],
                name=patient_data[1],
                age=patient_data[2],
                gender=patient_data[3],
                symptoms=patient_data[4],
                existing_conditions=patient_data[5],
                medications=patient_data[6],
                allergies=patient_data[7],
                vital_signs=patient_data[8],
                lab_results=patient_data[9],
                visit_date=patient_data[10]
            )
            self.patients[patient.patient_id] = patient

    def _initialize_medical_knowledge(self):
        knowledge_data = [
            MedicalKnowledge(
                condition="Pneumonia",
                description="Inflammation of the lungs typically caused by infection",
                common_symptoms=["fever", "cough", "shortness of breath", "chest pain"],
                risk_factors=["age > 65", "smoking", "immunosuppression", "chronic lung disease"],
                treatment_options=["Antibiotics", "Oxygen therapy", "Cough medicine"],
                urgency_level="HIGH"
            ),
            MedicalKnowledge(
                condition="Migraine",
                description="Recurrent headaches often with nausea and sensitivity to light",
                common_symptoms=["headache", "nausea", "photophobia", "aura"],
                risk_factors=["family history", "female gender", "stress"],
                treatment_options=["Pain relievers", "Triptans", "Preventive medications"],
                urgency_level="LOW"
            ),
            MedicalKnowledge(
                condition="Myocardial Infarction",
                description="Heart attack due to reduced blood flow to the heart",
                common_symptoms=["chest pain", "shortness of breath", "dizziness", "nausea"],
                risk_factors=["age", "hypertension", "high cholesterol", "smoking"],
                treatment_options=["Aspirin", "Nitroglycerin", "PCI", "Thrombolytics"],
                urgency_level="CRITICAL"
            ),
            MedicalKnowledge(
                condition="Gastroenteritis",
                description="Inflammation of the stomach and intestines",
                common_symptoms=["abdominal pain", "nausea", "vomiting", "diarrhea"],
                risk_factors=["Contaminated food/water", "Poor hygiene"],
                treatment_options=["Fluid replacement", "Anti-emetics", "Antidiarrheals"],
                urgency_level="LOW"
            ),
            MedicalKnowledge(
                condition="Sepsis",
                description="Life-threatening organ dysfunction caused by infection",
                common_symptoms=["fever", "chills", "confusion", "rapid heart rate"],
                risk_factors=["Elderly", "Immunocompromised", "Chronic illness"],
                treatment_options=["Antibiotics", "IV fluids", "Vasopressors"],
                urgency_level="CRITICAL"
            ),
        ]

        for knowledge in knowledge_data:
            self.medical_knowledge[knowledge.condition] = knowledge

    def identify_task(self, query: str) -> Dict:
        query_lower = query.lower()

        task_mapping = {
            'symptom': MedicalTaskType.SYMPTOM_CHECK,
            'symptoms': MedicalTaskType.SYMPTOM_CHECK,
            'diagnose': MedicalTaskType.DIAGNOSIS,
            'diagnosis': MedicalTaskType.DIAGNOSIS,
            'medication': MedicalTaskType.MEDICATION_CHECK,
            'drug': MedicalTaskType.MEDICATION_CHECK,
            'lab': MedicalTaskType.LAB_INTERPRETATION,
            'test': MedicalTaskType.LAB_INTERPRETATION,
            'result': MedicalTaskType.LAB_INTERPRETATION,
            'risk': MedicalTaskType.RISK_ASSESSMENT,
            'treatment': MedicalTaskType.TREATMENT_RECOMMENDATION,
            'urgent': MedicalTaskType.TRIAGE,
            'emergency': MedicalTaskType.TRIAGE,
            'triage': MedicalTaskType.TRIAGE,
        }

        for keyword, task_type in task_mapping.items():
            if keyword in query_lower:
                return {'task_type': task_type, 'confidence': 0.9}

        return {'task_type': MedicalTaskType.GENERAL, 'confidence': 0.6}

    def _get_patient(self, patient_id: str) -> Optional[Dict]:
        if patient_id in self.patients:
            return self.patients[patient_id].to_dict()
        return None

    def _get_condition_knowledge(self, condition: str) -> Optional[Dict]:
        if condition in self.medical_knowledge:
            return self.medical_knowledge[condition].__dict__
        return None

    def process_query(self, query: str) -> MedicalResponse:
        print(f"\n🏥 Medical Query: {query}")
        print("-" * 50)

        task_decision = self.identify_task(query)
        task_type = task_decision['task_type']
        confidence = task_decision['confidence']

        print(f"📋 Task Identification: {task_type.value} (confidence: {confidence:.2%})")

        # ✅ FIXED: Correct regex pattern for patient ID extraction
        patient_match = re.search(r'P\d{3}', query.upper())
        patient_id = patient_match.group(0) if patient_match else None

        conditions = list(self.medical_knowledge.keys())
        mentioned_condition = None
        for condition in conditions:
            if condition.lower() in query.lower():
                mentioned_condition = condition
                break

        data = {}
        action_taken = ""
        diagnosis = None
        referral_needed = False
        urgency = "LOW"

        if task_type == MedicalTaskType.DIAGNOSIS or task_type == MedicalTaskType.SYMPTOM_CHECK:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    action_taken = f"Retrieved patient {patient_id}"

                    matching_conditions = []
                    for condition, knowledge in self.medical_knowledge.items():
                        if any(symptom.lower() in condition.lower() for symptom in patient_data['symptoms']):
                            matching_conditions.append(condition)
                        elif any(symptom.lower() in ' '.join(knowledge.common_symptoms).lower()
                                for symptom in patient_data['symptoms']):
                            matching_conditions.append(condition)

                    if matching_conditions:
                        data['possible_conditions'] = matching_conditions
                        data['condition_details'] = [
                            self._get_condition_knowledge(c) for c in matching_conditions[:3]
                        ]
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['all_patients'] = [p.to_dict() for p in self.patients.values()]
                action_taken = "Retrieved all patients"

        elif task_type == MedicalTaskType.LAB_INTERPRETATION:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    data['lab_results'] = patient_data.get('lab_results', {})
                    action_taken = f"Retrieved lab results for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['lab_results'] = []
                for patient in self.patients.values():
                    data['lab_results'].append({
                        'patient_id': patient.patient_id,
                        'name': patient.name,
                        'results': patient.lab_results
                    })
                action_taken = "Retrieved lab results for all patients"

        elif task_type == MedicalTaskType.MEDICATION_CHECK:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    data['medications'] = patient_data.get('medications', [])
                    data['allergies'] = patient_data.get('allergies', [])
                    action_taken = f"Retrieved medication information for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['medications_summary'] = {}
                for patient in self.patients.values():
                    data['medications_summary'][patient.patient_id] = {
                        'name': patient.name,
                        'medications': patient.medications,
                        'allergies': patient.allergies
                    }
                action_taken = "Retrieved medication information for all patients"

        elif task_type == MedicalTaskType.RISK_ASSESSMENT:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    risk_factors = []
                    if patient_data['age'] > 65:
                        risk_factors.append("Age > 65")
                    if patient_data.get('existing_conditions'):
                        risk_factors.append("Existing conditions")
                    if patient_data.get('vital_signs', {}).get('bp'):
                        bp = patient_data['vital_signs']['bp']
                        if bp != "Normal":
                            risk_factors.append(f"Abnormal blood pressure: {bp}")
                    data['risk_factors'] = risk_factors
                    action_taken = f"Assessed risk factors for {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['risk_assessment'] = "Please specify a patient ID"
                action_taken = "Risk assessment requires patient ID"

        elif task_type == MedicalTaskType.TRIAGE:
            if patient_id:
                patient_data = self._get_patient(patient_id)
                if patient_data:
                    data['patient'] = patient_data
                    symptoms = patient_data.get('symptoms', [])

                    if any(s in ['chest pain', 'shortness of breath'] for s in symptoms):
                        urgency = "CRITICAL"
                        referral_needed = True
                    elif 'fever' in symptoms and any(t in symptoms for t in ['chills', 'muscle aches']):
                        urgency = "HIGH"
                    elif 'headache' in symptoms:
                        urgency = "MEDIUM"
                    else:
                        urgency = "LOW"

                    data['triage_level'] = urgency
                    action_taken = f"Triaged patient {patient_id}"
                else:
                    data['error'] = f"Patient {patient_id} not found"
                    action_taken = "Patient lookup failed"
            else:
                data['triage_summary'] = [
                    {'patient_id': p.patient_id, 'name': p.name, 'symptoms': p.symptoms}
                    for p in self.patients.values()
                ]
                action_taken = "Retrieved triage summary for all patients"

        else:
            data['message'] = "General medical inquiry received"
            action_taken = "General query processed"

        # Generate reasoning using Fable-5
        reasoning_prompt = self._generate_reasoning_prompt(query, task_type, data)
        fable_response = self.fable.query(reasoning_prompt)

        print(f"🧠 Fable-5 Reasoning: {fable_response.content[:200]}...")

        return MedicalResponse(
            task_type=task_type,
            action_taken=action_taken,
            data=data,
            reasoning=fable_response.content,
            diagnosis=mentioned_condition or "Further analysis needed",
            confidence=confidence,
            recommendations=self._extract_recommendations(fable_response.content),
            referral_needed=referral_needed,
            urgency=urgency,
            timestamp=datetime.now().isoformat()
        )

    def _generate_reasoning_prompt(self, query: str, task_type: MedicalTaskType, data: Dict) -> str:
        return f"""
        You are a Medical Diagnostics AI Assistant providing clinical decision support.

        User Query: "{query}"
        Task Type: {task_type.value}

        Available Medical Data:
        {json.dumps(data, indent=2, default=str)}

        Please provide:
        1. A clinical analysis of the situation
        2. Possible diagnoses based on the data
        3. Recommended next steps or tests
        4. Treatment recommendations
        5. Any red flags or urgent concerns
        6. A summary for the healthcare provider

        Your response should be professional, evidence-based, and actionable.
        Always include a disclaimer that this is AI-assisted decision support and not a substitute for professional medical judgment.
        """

    def _extract_recommendations(self, reasoning: str) -> List[str]:
        recommendations = []
        lines = reasoning.split('\n')
        for line in lines:
            if any(keyword in line.lower() for keyword in ['recommend', 'advise', 'suggest', 'should']):
                cleaned = line.strip().lstrip('•*-+0123456789. ')
                if len(cleaned) > 10 and cleaned not in recommendations:
                    recommendations.append(cleaned[:200])
        return recommendations[:5]

    def handle_query(self, query: str) -> str:
        response = self.process_query(query)

        output = []
        output.append("=" * 60)
        output.append("🏥 Medical Diagnostics Response")
        output.append("=" * 60)
        output.append(f"📋 Task: {response.task_type.value}")
        output.append(f"✅ Action: {response.action_taken}")
        output.append(f"🔒 Confidence: {response.confidence:.2%}")
        if response.diagnosis:
            output.append(f"🔍 Diagnosis: {response.diagnosis}")
        output.append(f"🚨 Urgency: {response.urgency}")
        if response.referral_needed:
            output.append("⚠️ Referral Needed: YES")
        output.append("-" * 60)
        output.append("📊 Clinical Analysis:")
        output.append(response.reasoning)
        if response.recommendations:
            output.append("-" * 60)
            output.append("💡 Recommendations:")
            for i, rec in enumerate(response.recommendations, 1):
                output.append(f"   {i}. {rec}")
        output.append("=" * 60)
        output.append("⚠️ DISCLAIMER: This is AI-assisted decision support.")
        output.append("   Always consult with a qualified healthcare professional.")
        output.append("   This system does not replace clinical judgment.")
        output.append("=" * 60)

        return "\n".join(output)


# ============================================================
# PART 4: DEMONSTRATION - ALL 5 QUERIES
# ============================================================

def run_medical_demo():
    print("\n" + "="*60)
    print("🏥  MEDICAL DIAGNOSTICS AGENT (Fable-5 Edition)")
    print("   Complete Agentic Solution for Clinical Decision Support")
    print("="*60 + "\n")

    agent = MedicalDiagnosticsAgent()

    # ✅ FIXED: ALL 5 test queries (including medication reconciliation)
    test_queries = [
        "Diagnose patient P001",
        "What are the lab results for P003?",
        "What medications is patient P002 taking?",
        "Assess risk factors for P003",
        "Triage patient P005",
    ]

    # ✅ FIXED: Run ALL 5 queries
    for i, query in enumerate(test_queries, 1):
        print(f"\n{'='*60}")
        print(f"TEST {i}")
        print('='*60)
        result = agent.handle_query(query)
        print(result)


# ============================================================
# PART 5: COMPARISON FUNCTION (OPTIONAL)
# ============================================================

def compare_engines():
    """Compare all three engines on the same queries."""

    print("\n" + "="*70)
    print("🏥  THREE-WAY ENGINE COMPARISON")
    print("   Inkling | Kimi-K3 | Fable-5")
    print("="*70 + "\n")

    # Import other engine classes (assuming they exist)
    try:
        from inkling_client import InklingClient
        from kimi_client import KimiClient
    except ImportError:
        print("⚠️ Other engine clients not available.")
        print("   Only Fable-5 will be tested.")
        run_medical_demo()
        return

    # Initialize all three engines
    agents = {
        "Inkling": MedicalDiagnosticsAgent(InklingClient()),
        "Kimi-K3": MedicalDiagnosticsAgent(KimiClient()),
        "Fable-5": MedicalDiagnosticsAgent(FableClient())
    }

    test_queries = [
        "Diagnose patient P001",
        "What are the lab results for P003?",
        "What medications is patient P002 taking?",
        "Assess risk factors for P003",
        "Triage patient P005",
    ]

    for query in test_queries:
        print(f"\n{'='*70}")
        print(f"🔍 QUERY: {query}")
        print("="*70)

        for name, agent in agents.items():
            print(f"\n📊 {name.upper()}")
            print("-" * 50)
            result = agent.handle_query(query)
            # Print only first 500 chars for comparison
            print(result[:500] + "...\n")


# ============================================================
# RUN DEMO
# ============================================================
if __name__ == "__main__":
    run_medical_demo()
    # Uncomment below for three-way comparison
    # compare_engines()

✅ API key loaded from Colab secrets! Ending: ****VgAA

🏥  MEDICAL DIAGNOSTICS AGENT (Fable-5 Edition)
   Complete Agentic Solution for Clinical Decision Support


🏥  MEDICAL DIAGNOSTICS AGENT (Gemma-4 & Fable-5)

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...
   ✅ Tokenizer loaded. Vocab size: 262144

👁️ Loading vision model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...
   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Fable-5 client initialized
   Model: claude-fable-5
✅ Medical Diagnostics Agent Ready!
   🏥 Patients: 5
   📚 Conditions: 5


TEST 1

🏥 Medical Query: Diagnose patient P001
--------------------------------------------------
📋 Task Identification: diagnosis (confidence: 90.00%)
🧠 Fable-5 Reasoning: # Clinical Decision Support Report — Patient P001

**Patient:** John Smith | 45M | Visit: 2026-08-04 15:00

---

## 1. Clinical Analysis

**Presentation:** 45-year-old male with fever (38.5°C), cough,...
🏥 Medical Diagnostics Response
📋 Task: diagnosis
✅ Action: Retrieved patient P001
🔒 Confidence: 90.00%
🔍 Diagnosis: Further analysis needed
🚨 Urgency: LOW
------------------------------------------